# Southend Pier -- Initial Tide-Level Modelling (PyTorch)

Baseline MLP/RNN/LSTM models predicting water level from its own recent history. No harmonic or meteorological inputs yet.

**Data**: `3_Cleaned/10_SOUTHEND_chatter_and_stuck_patched.csv`, 10-minute readings, 2004-2024, chatter/stuck-sensor faults already patched.

**Task**: given the last 16h (96 steps), predict the next 10-minute value.

**Split**: train 2004-2017, validation 2018-2020, test 2021-2024. Scaler and models fit on train only; test touched once at the end.

**Models**: MLP, vanilla RNN, LSTM.

In [1]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn


## 1. Setup


In [2]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"BF16 supported:  {torch.cuda.is_bf16_supported()}")
else:
    print("No GPU detected -- running on CPU. That's fine at this notebook's scale (see intro); "
          "LOOKBACK and TRAIN_STRIDE below are chosen to keep CPU training fast.")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SEED = 42
LOOKBACK = 96       # 16h of 10-minute steps (>1 full semi-diurnal cycle). Raise this (e.g. 288 = 48h)
                    # if running on a GPU -- RNN/LSTM cost scales with LOOKBACK, and 288 takes hours on CPU.
TRAIN_STRIDE = 3    # only start a new training window every 3rd step (30 min) to cut epoch time on CPU;
                    # validation/test stay at full 10-min resolution so reported metrics aren't affected.
BATCH_SIZE = 1024
EPOCHS = 60
PATIENCE = 10
LR = 1e-3
TRAIN_END = '2018-01-01'
VAL_END = '2021-01-01'

LOAD_FROM_CHECKPOINT = True  # skip training and load saved weights from OUTPUT_DIR if present

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
torch.backends.cudnn.benchmark = True

PyTorch version: 2.11.0+cu128
CUDA available:  True
GPU:             NVIDIA A100-SXM4-80GB
BF16 supported:  True


In [3]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/tidal_analysis_and_prediction'
    DATA_PATH = os.path.join(BASE_DIR, '10_SOUTHEND_chatter_and_stuck_patched.csv')
    OUTPUT_DIR = os.path.join(BASE_DIR, 'initial_outputs')
else:
    DATA_PATH = '../3_Cleaned/10_SOUTHEND_chatter_and_stuck_patched.csv'
    OUTPUT_DIR = '../outputs/initial_outputs'

os.makedirs(OUTPUT_DIR, exist_ok=True)
assert os.path.exists(DATA_PATH), f"Can't find data at {DATA_PATH} -- update DATA_PATH above."

import sys
if IN_COLAB:
    sys.path.insert(0, BASE_DIR)  # plot_config.py uploaded to the same Drive folder as the data
from plot_config import SERIES_COLOURS, save_fig, set_report_dir
set_report_dir(os.path.join(BASE_DIR, 'report_images') if IN_COLAB else '../report_images')

Mounted at /content/drive


## 2. Load & inspect data


In [4]:
usecols = ['DateTime', 'Observed_ODN', 'is_chatter_flagged', 'is_stuck_flagged', 'is_imputed']
df = pd.read_csv(DATA_PATH, usecols=usecols, parse_dates=['DateTime'])
df = df.sort_values('DateTime').reset_index(drop=True)
df['is_flagged'] = df['is_chatter_flagged'] | df['is_stuck_flagged'] | df['is_imputed']

STEP = pd.Timedelta('10min')
gap_mask = df['DateTime'].diff() > STEP

print(f"{len(df):,} rows, {df.DateTime.min()} to {df.DateTime.max()}")
print(f"{gap_mask.sum()} gaps > 10min ({gap_mask.sum() + 1} contiguous segments)")
print(f"Flagged (chatter/stuck/imputed) points: {df['is_flagged'].sum():,} ({100 * df['is_flagged'].mean():.3f}%)")
df.head()


1,103,956 rows, 2004-01-01 00:00:00 to 2024-12-31 23:50:00
90 gaps > 10min (91 contiguous segments)
Flagged (chatter/stuck/imputed) points: 9,975 (0.904%)


,DateTime,is_imputed,Observed_ODN,is_chatter_flagged,is_stuck_flagged,is_flagged
0,2004-01-01 00:00:00,False,-1.819,False,False,False
1,2004-01-01 00:10:00,False,-1.855,False,False,False
2,2004-01-01 00:20:00,False,-1.874,False,False,False
3,2004-01-01 00:30:00,False,-1.895,False,False,False
4,2004-01-01 00:40:00,False,-1.904,False,False,False


## 3. Chronological train / validation / test split

Three non-overlapping blocks, split before windowing so no window crosses a boundary.

In [5]:
train_df = df[df['DateTime'] < TRAIN_END].reset_index(drop=True)
val_df = df[(df['DateTime'] >= TRAIN_END) & (df['DateTime'] < VAL_END)].reset_index(drop=True)
test_df = df[df['DateTime'] >= VAL_END].reset_index(drop=True)

for split_name, split in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f"{split_name:5s}: {len(split):>8,} rows  [{split.DateTime.min()} -> {split.DateTime.max()}]  "
          f"flagged={100 * split['is_flagged'].mean():.3f}%")


train:  735,760 rows  [2004-01-01 00:00:00 -> 2017-12-31 23:50:00]  flagged=1.338%
val  :  157,824 rows  [2018-01-01 00:00:00 -> 2020-12-31 23:50:00]  flagged=0.038%
test :  210,372 rows  [2021-01-01 00:00:00 -> 2024-12-31 23:50:00]  flagged=0.032%


## 4. Windowing

Sliding windows of `LOOKBACK` past values -> next value, built within each split and gap-free segment. Training windows subsampled by `TRAIN_STRIDE`; validation/test kept at full resolution.

In [6]:
def build_windows(sub_df, lookback=LOOKBACK):
    sub_df = sub_df.reset_index(drop=True)
    is_gap = sub_df['DateTime'].diff() > STEP
    seg_id = is_gap.cumsum().to_numpy()

    values = sub_df['Observed_ODN'].to_numpy(dtype=np.float32)
    flagged = sub_df['is_flagged'].to_numpy()
    times = sub_df['DateTime'].to_numpy()

    X_list, y_list, flag_list, time_list = [], [], [], []
    for seg in np.unique(seg_id):
        idx = np.where(seg_id == seg)[0]
        if len(idx) < lookback + 1:
            continue
        windows = np.lib.stride_tricks.sliding_window_view(values[idx], lookback + 1)
        X_list.append(windows[:, :lookback])
        y_list.append(windows[:, lookback])
        flag_list.append(flagged[idx][lookback:])
        time_list.append(times[idx][lookback:])

    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    flag = np.concatenate(flag_list, axis=0)
    time = np.concatenate(time_list, axis=0)
    return X, y, flag, time


In [7]:
X_train, y_train, flag_train, time_train = build_windows(train_df)
X_val, y_val, flag_val, time_val = build_windows(val_df)
X_test, y_test, flag_test, time_test = build_windows(test_df)

X_train, y_train, flag_train, time_train = (
    X_train[::TRAIN_STRIDE], y_train[::TRAIN_STRIDE], flag_train[::TRAIN_STRIDE], time_train[::TRAIN_STRIDE]
)

print(f"windows -> train: {X_train.shape} (stride {TRAIN_STRIDE}), val: {X_val.shape}, test: {X_test.shape}")


windows -> train: (243001, 96) (stride 3), val: (157728, 96), test: (209389, 96)


## 5. Scaling

Standardised using the train split's mean/std only.

In [8]:
train_mean = float(train_df['Observed_ODN'].mean())
train_std = float(train_df['Observed_ODN'].std())
print(f"train mean={train_mean:.4f} m, std={train_std:.4f} m")

def to_tensor(a):
    return torch.tensor(a, dtype=torch.float32, device=device)

X_train_t = to_tensor((X_train - train_mean) / train_std)
y_train_t = to_tensor((y_train - train_mean) / train_std)
X_val_t = to_tensor((X_val - train_mean) / train_std)
y_val_t = to_tensor((y_val - train_mean) / train_std)
X_test_t = to_tensor((X_test - train_mean) / train_std)

flag_train_t = torch.tensor(flag_train, dtype=torch.bool, device=device)
flag_val_t = torch.tensor(flag_val, dtype=torch.bool, device=device)

train mean=0.1459 m, std=1.5730 m


## 6. Baselines

Persistence baseline: `y(t+1) = y(t)`, flagged/imputed targets masked out.

In [9]:
def masked_rmse_mae(pred, true, flagged):
    mask = ~flagged
    err = pred[mask] - true[mask]
    return float(np.sqrt(np.mean(err ** 2))), float(np.mean(np.abs(err)))

def persistence_pred(X_raw):
    return X_raw[:, -1]

results = []
for split_name, X_raw, y_raw, flg in [('val', X_val, y_val, flag_val), ('test', X_test, y_test, flag_test)]:
    rmse, mae = masked_rmse_mae(persistence_pred(X_raw), y_raw, flg)
    results.append({'model': 'Persistence', 'split': split_name, 'rmse_m': rmse, 'mae_m': mae})

pd.DataFrame(results).pivot(index='model', columns='split', values=['rmse_m', 'mae_m']).round(4)

rmse_m           mae_m        
split          test     val    test     val
model                                      
Persistence  0.1329  0.1351  0.1181  0.1202

## 6b. UTide harmonic baseline

[UTide](https://github.com/wesleybowman/UTide) fits tidal constituents from observed data and reconstructs the tide at any timestamp. Fit on train+val (2004-2020), evaluated on test (2021-2024). Flagged points excluded from the fit.

In [11]:
try:
    import utide
except ImportError:
    import sys
    !{sys.executable} -m pip install -q UTide
    import utide


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.2/75.2 kB 8.6 MB/s eta 0:00:00


In [12]:
import utide

site_lat = 51.5145  # Southend Pier, matches EDA/utide_test.ipynb

# Fit on train+val, excluding flagged (chatter/stuck/imputed) points to avoid
# fitting on UTide's own past reconstruction.
fit_df = pd.concat([train_df, val_df], ignore_index=True)
fit_df.loc[fit_df['is_flagged'], 'Observed_ODN'] = np.nan

coef = utide.solve(fit_df['DateTime'], fit_df['Observed_ODN'],
                    lat=site_lat, method='ols', conf_int='none')

# Reconstruct at the same (windowed) test timestamps/targets used by every
# other model in the results table, for a like-for-like row.
tide_test = utide.reconstruct(pd.DatetimeIndex(time_test), coef, verbose=False)
rmse_u, mae_u = masked_rmse_mae(tide_test.h, y_test, flag_test)
results.append({'model': 'UTide', 'split': 'test', 'rmse_m': rmse_u, 'mae_m': mae_u})

print(f"UTide test: rmse={rmse_u:.4f} m, mae={mae_u:.4f} m")

solve: matrix prep ... solution ... done.
UTide test: rmse=0.2427 m, mae=0.1754 m


## 7. Models

MLP, RNN, LSTM -- single-feature, `(batch, LOOKBACK)` in, `(batch,)` out.

In [13]:
class MLP(nn.Module):
    def __init__(self, lookback, hidden=(128, 64)):
        super().__init__()
        dims = [lookback, *hidden]
        layers = []
        for i in range(len(dims) - 1):
            layers += [nn.Linear(dims[i], dims[i + 1]), nn.ReLU()]
        layers.append(nn.Linear(dims[-1], 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class RNNModel(nn.Module):
    def __init__(self, hidden_size=64, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(input_size=1, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x.unsqueeze(-1))
        return self.head(out[:, -1, :]).squeeze(-1)


class LSTMModel(nn.Module):
    def __init__(self, hidden_size=64, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x.unsqueeze(-1))
        return self.head(out[:, -1, :]).squeeze(-1)


## 8. Training utilities

Batch iterator plus a training loop with early stopping on validation loss. Flagged targets are masked out of both train and val loss.

In [14]:
def batch_iter(X, y, flag, batch_size, shuffle):
    n = X.shape[0]
    idx = torch.randperm(n, device=X.device) if shuffle else torch.arange(n, device=X.device)
    for start in range(0, n, batch_size):
        b = idx[start:start + batch_size]
        yield X[b], y[b], flag[b]


def masked_mse(pred, true, flag_b):
    per_sample = (pred - true) ** 2
    return per_sample[~flag_b].mean()


def train_model(model, X_train, y_train, flag_train, X_val, y_val, flag_val, epochs, batch_size, lr, patience):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
    use_amp = device.type == 'cuda'

    best_val = float('inf')
    best_state = None
    epochs_no_improve = 0
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(epochs):
        model.train()
        running_loss, n_seen = 0.0, 0
        for xb, yb, flag_b in batch_iter(X_train, y_train, flag_train, batch_size, shuffle=True):
            opt.zero_grad()
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=use_amp):
                loss = masked_mse(model(xb), yb, flag_b)
            loss.backward()
            opt.step()
            n_valid = int((~flag_b).sum())
            running_loss += loss.item() * n_valid
            n_seen += n_valid
        train_loss = running_loss / n_seen

        model.eval()
        val_loss_sum, n_val = 0.0, 0
        with torch.no_grad():
            for xb, yb, flag_b in batch_iter(X_val, y_val, flag_val, batch_size, shuffle=False):
                with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=use_amp):
                    loss = masked_mse(model(xb), yb, flag_b)
                n_valid = int((~flag_b).sum())
                val_loss_sum += loss.item() * n_valid
                n_val += n_valid
        val_loss = val_loss_sum / n_val

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        sched.step(val_loss)

        improved = val_loss < best_val - 1e-6
        if improved:
            best_val = val_loss
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        print(f"  epoch {epoch + 1:03d}  train_loss={train_loss:.5f}  val_loss={val_loss:.5f}"
              f"  lr={opt.param_groups[0]['lr']:.1e}{'  *' if improved else ''}")

        if epochs_no_improve >= patience:
            print(f"  early stopping at epoch {epoch + 1} (best val_loss={best_val:.5f})")
            break

    model.load_state_dict(best_state)
    return model, history


@torch.no_grad()
def compute_metrics(model, X_t, y_raw, flagged, batch_size=8192):
    model.eval()
    preds = []
    for start in range(0, X_t.shape[0], batch_size):
        preds.append(model(X_t[start:start + batch_size]))
    preds_m = torch.cat(preds).cpu().numpy() * train_std + train_mean
    rmse, mae = masked_rmse_mae(preds_m, y_raw, flagged)
    return rmse, mae, preds_m


def load_or_train(model, checkpoint_path, train_fn, label):
    # Skip training and load a saved state dict if LOAD_FROM_CHECKPOINT is set and one exists --
    # architecture is fixed by LOOKBACK/hidden sizes above, not swept anywhere in this notebook,
    # so re-instantiating the same class is enough to make a saved state dict load cleanly.
    if LOAD_FROM_CHECKPOINT and os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        model.eval()
        print(f"[{label}] loaded weights from {checkpoint_path}")
        return model, None
    return train_fn(model)

## 9. Train MLP, RNN, LSTM

Same training loop and seed for each model.

In [ ]:
model_factories = {
    'MLP': lambda: MLP(LOOKBACK),
    'RNN': lambda: RNNModel(hidden_size=64, num_layers=1),
    'LSTM': lambda: LSTMModel(hidden_size=64, num_layers=1),
}

trained = {}
histories = {}
for name, factory in model_factories.items():
    set_seed()
    model = factory().to(device)
    checkpoint_path = os.path.join(OUTPUT_DIR, f'initial_{name.lower()}.pt')

    def _train(m, name=name):
        print(f"\n=== Training {name} ===")
        return train_model(
            m, X_train_t, y_train_t, flag_train_t, X_val_t, y_val_t, flag_val_t,
            epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patience=PATIENCE,
        )

    model, history = load_or_train(model, checkpoint_path, _train, label=name)
    trained[name] = model
    histories[name] = history

[MLP] loaded weights from /content/drive/MyDrive/tidal_analysis_and_prediction/initial_outputs/initial_mlp.pt
[RNN] loaded weights from /content/drive/MyDrive/tidal_analysis_and_prediction/initial_outputs/initial_rnn.pt


## 10. Results

RMSE/MAE (m) on validation and test, flagged targets excluded, alongside the baselines.

In [ ]:
val_preds, test_preds = {}, {}
for name, model in trained.items():
    rmse_v, mae_v, val_preds[name] = compute_metrics(model, X_val_t, y_val, flag_val)
    rmse_t, mae_t, test_preds[name] = compute_metrics(model, X_test_t, y_test, flag_test)
    results.append({'model': name, 'split': 'val', 'rmse_m': rmse_v, 'mae_m': mae_v})
    results.append({'model': name, 'split': 'test', 'rmse_m': rmse_t, 'mae_m': mae_t})

results_df = pd.DataFrame(results)
results_df.pivot(index='model', columns='split', values=['rmse_m', 'mae_m']).round(4)


## 11. Loss curves


In [ ]:
available_histories = {name: h for name, h in histories.items() if h is not None}
if not available_histories:
    print("All models loaded from checkpoint -- no training history to plot.")
else:
    n = len(available_histories)
    fig, axes = plt.subplots(2, n, figsize=(5 * n, 8), squeeze=False)
    for col, (name, history) in enumerate(available_histories.items()):
        epochs = np.arange(1, len(history['train_loss']) + 1)
        for row, start_epoch in enumerate([1, 2]):
            ax = axes[row, col]
            mask = epochs >= start_epoch
            ax.plot(epochs[mask], np.array(history['train_loss'])[mask], label='train')
            ax.plot(epochs[mask], np.array(history['val_loss'])[mask], label='val')
            ax.set_yscale('log')
            ax.set_xlabel('epoch')
            if col == 0:
                ax.legend(fontsize=8)
            ax.grid(alpha=0.3, which='both')
        axes[0, col].set_title(name, fontsize=11)

    axes[0, 0].set_ylabel('MSE loss\n(log scale)', fontsize=9)
    axes[1, 0].set_ylabel('MSE loss\n(log, epoch ≥ 2)', fontsize=9)
    save_fig(fig, 'initial_loss_curves', width_in=6.3, height_in=4.4, w_pad=2.0, h_pad=2.5)
    plt.show()

## 11b. Why is val loss lower than train loss?

Val is an easier period, not necessarily better generalisation: `train_df` has ~35x the flagged rate of `val_df` (1.34% vs 0.04%), and flagged points are the hardest to fit.

In [ ]:
splits = {'train': train_df, 'val': val_df, 'test': test_df}
step_diffs = {}

print(f"{'split':6s} {'n':>10s} {'mean_m':>8s} {'std_m':>8s} {'min_m':>8s} {'max_m':>8s} "
      f"{'step_mean_abs':>14s} {'step_std':>10s} {'flagged%':>9s}")
for name, d in splits.items():
    level = d['Observed_ODN']

    # genuine 10-min step, both endpoints unflagged -- diff() on the raw series first so we
    # never treat "one removed/flagged row apart" or "either side of a data gap" as a real step
    is_real_step = (d['DateTime'].diff() == STEP) & ~d['is_flagged'] & ~d['is_flagged'].shift(1, fill_value=True)
    step = d['Observed_ODN'].diff()[is_real_step]
    step_diffs[name] = step

    print(f"{name:6s} {len(d):>10,} {level.mean():>8.4f} {level.std():>8.4f} {level.min():>8.4f} {level.max():>8.4f} "
          f"{step.abs().mean():>14.5f} {step.std():>10.5f} {100 * d['is_flagged'].mean():>8.3f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name, d in splits.items():
    axes[0].hist(d['Observed_ODN'], bins=100, histtype='step', density=True, label=name, linewidth=1.5)
axes[0].set_title('Distribution of Observed_ODN (level)', fontsize=11)
axes[0].set_xlabel('m ODN')
axes[0].set_ylabel('density')
axes[0].legend()
axes[0].grid(alpha=0.3)

for name, step in step_diffs.items():
    axes[1].hist(step, bins=200, range=(-0.15, 0.15), histtype='step', density=True, label=name, linewidth=1.5)
axes[1].set_title('Distribution of 10-min step-to-step change', fontsize=11)
axes[1].set_xlabel('Δ Observed_ODN (m)')
axes[1].set_yscale('log')
axes[1].legend()
axes[1].grid(alpha=0.3, which='both')

plt.tight_layout()
plt.show()

In [ ]:
is_real_step_full = (df['DateTime'].diff() == STEP) & ~df['is_flagged'] & ~df['is_flagged'].shift(1, fill_value=True)
full_step = df['Observed_ODN'].diff()[is_real_step_full]
full_year = df.loc[is_real_step_full, 'DateTime'].dt.year

yearly_step_std = full_step.groupby(full_year).std()

train_end_year = pd.Timestamp(TRAIN_END).year
val_end_year = pd.Timestamp(VAL_END).year
split_colors = {'train': '#d62728', 'val': '#1f77b4', 'test': '#2ca02c'}
bar_colors = [
    split_colors['train'] if y < train_end_year else split_colors['val'] if y < val_end_year else split_colors['test']
    for y in yearly_step_std.index
]

plt.figure(figsize=(14, 4))
plt.bar(yearly_step_std.index, yearly_step_std.values, color=bar_colors)
plt.axvline(train_end_year - 0.5, color='k', linestyle='--', linewidth=1)
plt.axvline(val_end_year - 0.5, color='k', linestyle='--', linewidth=1)
plt.ylabel('std of 10-min step change (m)')
plt.title('Year-by-year volatility -- train (red) / val (blue) / test (green)', fontsize=11)
plt.grid(alpha=0.3, axis='y')
plt.show()

## 12. Predictions vs. observed -- test set sample

One-step-ahead predictions over a sample window. Differences from the observed line are subtle at this horizon.

In [ ]:
window_start = pd.Timestamp('2022-01-01')
window_end = window_start + pd.Timedelta(days=14)
plot_mask = (time_test >= np.datetime64(window_start)) & (time_test < np.datetime64(window_end))

fig, (ax_level, ax_err) = plt.subplots(2, 1, figsize=(14, 6.5), sharex=True, height_ratios=[2, 1])

ax_level.plot(time_test[plot_mask], y_test[plot_mask], label='Observed', color='black', linewidth=1)
for name, preds_m in test_preds.items():
    ax_level.plot(time_test[plot_mask], preds_m[plot_mask], label=name, linewidth=1, color=SERIES_COLOURS.get(name))
ax_level.set_ylabel('Water level (m ODN)')
ax_level.legend()
ax_level.grid(alpha=0.3)

# Predictions sit within ~1-2cm of Observed at this horizon (see Section 10's RMSE/MAE), so on
# the raw water-level axis above every line overlaps -- whichever model is drawn last visually
# covers the rest. Plot the error against Observed instead, where the differences are visible.
for name, preds_m in test_preds.items():
    ax_err.plot(time_test[plot_mask], preds_m[plot_mask] - y_test[plot_mask], linewidth=1, color=SERIES_COLOURS.get(name))
ax_err.axhline(0, color='black', linewidth=0.8)
ax_err.set_xlabel('Date')
ax_err.set_ylabel('Error (m)')
ax_err.grid(alpha=0.3)

save_fig(fig, 'initial_predictions_vs_observed', width_in=6.3, height_in=4.2)
plt.show()

## 13. Recursive multi-step rollout evaluation

Section 10's RMSE is one-step-ahead (10min): every prediction is fed the true last `LOOKBACK` observations. Real forecasting needs longer horizons, so each neural model is rolled forward recursively here -- its own prediction is appended to the input window and fed back in as the newest observation, instead of the true value -- out to 48h, then scored at 10min/1h/6h/24h/48h against the observed series.

`Persistence`'s recursive rollout is just a flat continuation of the last observed value -- it has no mechanism to update. `UTide` needs no rollout at all: as a fixed harmonic model it's evaluated directly at each target timestamp regardless of which window it falls in.

In [ ]:
HORIZONS = {'10min': 1, '1h': 6, '6h': 36, '24h': 144, '48h': 288}  # steps of STEP (10min) ahead
MAX_HORIZON = max(HORIZONS.values())
ROLLOUT_STRIDE = 144  # new rollout start point every 24h across the test set -- trades off
                       # RMSE estimate variance against compute (288 recursive steps/window)

test_values = test_df['Observed_ODN'].to_numpy(dtype=np.float32)
test_flagged = test_df['is_flagged'].to_numpy()
test_seg_id = (test_df['DateTime'].diff() > STEP).cumsum().to_numpy()

# valid start index i: LOOKBACK history before i and MAX_HORIZON steps after i, all within one
# gap-free segment (seg_id is non-decreasing, so equal endpoints imply constant in between)
candidates = np.arange(LOOKBACK, len(test_df) - MAX_HORIZON + 1)
same_segment = test_seg_id[candidates - LOOKBACK] == test_seg_id[candidates + MAX_HORIZON - 1]
rollout_start_idx = candidates[same_segment][::ROLLOUT_STRIDE]
print(f"{len(rollout_start_idx):,} rollout start points across the test set "
      f"(every {ROLLOUT_STRIDE * 10}min, {MAX_HORIZON * 10}min horizon each)")


@torch.no_grad()
def recursive_rollout(model, start_idx, values, horizon_steps, batch_size=2048):
    """Roll a model forward `horizon_steps` steps from each start index, feeding each
    prediction back in as the newest input instead of the true next value."""
    model.eval()
    n = len(start_idx)
    preds = np.empty((n, horizon_steps), dtype=np.float32)
    for b0 in range(0, n, batch_size):
        b_idx = start_idx[b0:b0 + batch_size]
        hist = np.stack([values[i - LOOKBACK:i] for i in b_idx])
        hist_t = to_tensor((hist - train_mean) / train_std)
        for step in range(horizon_steps):
            next_scaled = model(hist_t)
            preds[b0:b0 + len(b_idx), step] = (next_scaled * train_std + train_mean).cpu().numpy()
            hist_t = torch.cat([hist_t[:, 1:], next_scaled.unsqueeze(-1)], dim=1)
    return preds

In [ ]:
rollout_preds = {}
for name, model in trained.items():
    rollout_preds[name] = recursive_rollout(model, rollout_start_idx, test_values, MAX_HORIZON)
    print(f"[{name}] rollout done")

# Persistence: flat continuation of the last observed value -- no mechanism to update
rollout_preds['Persistence'] = np.repeat(test_values[rollout_start_idx - 1][:, None], MAX_HORIZON, axis=1)

# UTide: fixed harmonic model, evaluated directly at each target timestamp -- no recursion needed
tide_test_full = utide.reconstruct(pd.DatetimeIndex(test_df['DateTime']), coef, verbose=False)
rollout_preds['UTide'] = np.stack(
    [tide_test_full.h[rollout_start_idx + h - 1] for h in range(1, MAX_HORIZON + 1)], axis=1
)

rollout_rows = []
for name, preds in rollout_preds.items():
    for label, h in HORIZONS.items():
        target_idx = rollout_start_idx + h - 1
        rmse, _ = masked_rmse_mae(preds[:, h - 1], test_values[target_idx], test_flagged[target_idx])
        rollout_rows.append({'model': name, 'horizon': label, 'rmse_m': rmse})

rollout_df = pd.DataFrame(rollout_rows)
rollout_table = rollout_df.pivot(index='model', columns='horizon', values='rmse_m')[list(HORIZONS)].round(4)
rollout_table = rollout_table.reindex(['LSTM', 'MLP', 'RNN', 'Persistence', 'UTide'])

rollout_path = os.path.join(OUTPUT_DIR, 'initial_rollout_rmse.csv')
rollout_table.to_csv(rollout_path)
print(f"saved {rollout_path}")
rollout_table

## 14. Next steps

- Direct multi-step output heads
- Add predictors: other gauges, meteorological data
- Sweep `LOOKBACK`, hidden sizes, layers; try GRU
- Check performance around storm surge periods

## 15. Save outputs to Google Drive

Saves this notebook's tables, model checkpoints, and figures: metrics/checkpoints to `OUTPUT_DIR`, dissertation-style PDF figures to `report_images/` (via `plot_config.save_fig`). Set `LOAD_FROM_CHECKPOINT = False` in Setup to force retraining instead of loading saved weights.

In [ ]:
# Section 10's val/test comparison table (Persistence, UTide, MLP, RNN, LSTM)
metrics_path = os.path.join(OUTPUT_DIR, 'initial_metrics.csv')
results_df.to_csv(metrics_path, index=False)
print(f"saved {metrics_path}")

# Model checkpoints -- state dict only, architecture is fixed by LOOKBACK/hidden sizes above,
# not swept anywhere in this notebook, so re-instantiating the same class is enough to load.
for name, model in trained.items():
    checkpoint_path = os.path.join(OUTPUT_DIR, f'initial_{name.lower()}.pt')
    torch.save({'model_state_dict': model.state_dict()}, checkpoint_path)
    print(f"saved {checkpoint_path}")

In [ ]:
import glob

print(f"\nAll initial-modelling outputs are under: {OUTPUT_DIR}")
for p in sorted(glob.glob(os.path.join(OUTPUT_DIR, '*'))):
    print(f"  {os.path.basename(p)}")